
###  AI geo validation core
###  Takes a raw value, ask the LLM, return a structured result.


> Imports

In [0]:
import json
import logging
from datetime import datetime
import requests
from pyspark.sql.types import FloatType, StringType, StructField, StructType

In [0]:
logger = logging.getLogger("geo_validator")

RESULT_SCHEMA = StructType([
    StructField("raw_value",        StringType(), True),
    StructField("corrected_value",  StringType(), True),
    StructField("confidence_score", FloatType(),  True),
    StructField("flag",             StringType(), True), 
    StructField("reason",           StringType(), True),
])

> Prompts

In [0]:
_SYSTEM = (
    "You are a geo data quality expert. "
    "Fix typos in geographic field values. "
    "Always respond with ONLY a valid JSON object — no extra text."
)

_USER = """Field type : {field_type}
Raw value  : "{raw_value}"
Known clean examples: {reference}

Return exactly this JSON:
{{
  "corrected_value"  : "<corrected text, identical to input if already correct>",
  "confidence_score" : <float 0.0–1.0>,
  "flag"             : "<CLEAN if ≥0.85 | FUZZY if 0.60–0.84 | MANUAL if <0.60>",
  "reason"           : "<one short sentence>"
}}"""

> Geo Validator

In [0]:
class GeoValidator:

    def __init__(self, token: str, workspace_url: str,
                 model: str = "databricks-meta-llama-3-3-70b-instruct"):
        self.token    = token
        self.base_url = workspace_url.rstrip("/")
        self.model    = model


# Validation method def
    def validate(self, raw_value: str, field_type: str,
                 reference: list) -> dict:
        """Ask the LLM to validate/correct one value. Returns a dict."""

        # Skip empty values
        if not raw_value or str(raw_value).strip() == "":
            return self._result(raw_value, raw_value, 1.0, "CLEAN", "Empty — skipped")

        prompt = _USER.format(
            field_type = field_type,
            raw_value  = raw_value,
            reference  = json.dumps(reference[:15]),
        )

        try:
            url  = f"{self.base_url}/serving-endpoints/{self.model}/invocations"
            resp = requests.post(
                url,
                headers={"Authorization": f"Bearer {self.token}",
                         "Content-Type": "application/json"},
                json={
                    "messages": [
                        {"role": "system", "content": _SYSTEM},
                        {"role": "user",   "content": prompt},
                    ],
                    "max_tokens":  200,
                    "temperature": 0.0,
                },
                timeout=30,
            )
            resp.raise_for_status()

            text = resp.json()["choices"][0]["message"]["content"].strip()

            # Strip markdown fence if the model adds
            if "```" in text:
                text = text.split("```")[1]
                if text.startswith("json"):
                    text = text[4:]
            text = text.strip()

            p = json.loads(text)
            return self._result(
                raw_value,
                str(p.get("corrected_value", raw_value)),
                float(p.get("confidence_score", 0.0)),
                str(p.get("flag", "MANUAL")),
                str(p.get("reason", "")),
            )

        except Exception as e:
            logger.warning(f"LLM error for '{raw_value}': {e}")
            return self._result(raw_value, raw_value, 0.0, "MANUAL",
                                f"Error: {str(e)[:100]}")

    @staticmethod
    def _result(raw, corrected, confidence, flag, reason):
        return {
            "raw_value":        raw,
            "corrected_value":  corrected,
            "confidence_score": confidence,
            "flag":             flag,
            "reason":           reason,
        }


In [0]:
# Get token and workspace URL from notebook context
token         = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
workspace_url = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()

# Test one value — should return a correction for 'Mumbay'
v = GeoValidator(token=token, workspace_url=workspace_url)
result = v.validate("Mumbay", "city", ["Mumbai", "Delhi", "Chennai", "Kolkata"])

print("Test result:")
for k, val in result.items():
    print(f"  {k:<20}: {val}")

# Expected:
#   raw_value           : Mumbay
#   corrected_value     : Mumbai
#   confidence_score    : 0.97  (or similar high value)
#   flag                : CLEAN
#   reason              : Common misspelling of Mumbai